# Fine-tune multilingual-E5-large v2 — Kaggle

Tréning na **hard negatives mineovaných pomocou e5-large-finetuned v1** (iterative hard negative mining).

### Pred spustením:
1. **Settings → Accelerator → GPU T4 x1**
2. Nahraj `data/hard_negatives_v2.jsonl` ako Kaggle dataset pomenovaný `ct26-hard-negatives-v2`
3. *Add Data* → tvoj dataset → potvrď
4. Spusti všetky bunky

### Výstup:
- `/kaggle/working/e5-large-finetuned-v2/final/` — sentence-transformers model
- Stiahni z **Output tab** a nahraj ako Kaggle dataset `ct26-e5-finetuned-v2`
- V pipeline notebooku nastav `DENSE_MODEL` na nový dataset

In [ ]:
# ── 0. GPU check ──────────────────────────────────────────────────────────────
import subprocess
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else 'No GPU!')
except FileNotFoundError:
    print('GPU nie je zapnuté! Zapni: Settings → Accelerator → GPU T4 x1')

In [ ]:
# ── 1. Install + restart kernel ───────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'sentence-transformers', 'transformers',
])
print('Reštartujem kernel...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# ?? 2. Config ?????????????????????????????????????????????????????????????????
import torch
from pathlib import Path

N_GPUS = torch.cuda.device_count()
print(f'Dostupn? GPU: {N_GPUS}x  ({[torch.cuda.get_device_name(i) for i in range(N_GPUS)]})')

DATA_PATH  = Path('/kaggle/input/ct26-hard-negatives-v2/hard_negatives_v2.jsonl')
OUTPUT_DIR = Path('/kaggle/working/e5-large-finetuned-v2')

# Pokra?uj z v1 checkpointu (ni??? LR = refinement, nie full retrain)
# Zme? na 'intfloat/multilingual-e5-large' pre tr?ning od base
MODEL_NAME = '/kaggle/input/ct26-e5-finetuned/final'

EPOCHS              = 2
BATCH_SIZE          = 32 if N_GPUS >= 2 else 16   # 2x T4: 32 (16/GPU), 1x T4: 16
GRAD_ACCUM          = 1  if N_GPUS >= 2 else 2    # efekt?vny batch = 32 v oboch pr?padoch
LR                  = 5e-6   # ni??? LR ? refinement z v1
ANCHOR_MAX_LEN      = 96
PASSAGE_MAX_LEN     = 128
NEGATIVES_PER_QUERY = 1
SEED                = 42
LOG_EVERY           = 20

# Volite?ne kombinuj s p?vodn?mi hard negatives (v1)
EXTRA_DATA = Path('/kaggle/input/ct26-hard-negatives/hard_negatives.jsonl')
USE_EXTRA  = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_PATH.exists(), f'Dataset nen?jden?: {DATA_PATH}\nPridaj ho cez Add Data'
assert NEGATIVES_PER_QUERY >= 1, 'NEGATIVES_PER_QUERY mus? by? >= 1'
print(f'Data:        {DATA_PATH}  ({DATA_PATH.stat().st_size/1e6:.1f} MB)')
print(f'Model:       {MODEL_NAME}')
print(f'Batch size:  {BATCH_SIZE}  (grad_accum={GRAD_ACCUM} ? efekt?vny batch={BATCH_SIZE*GRAD_ACCUM})')
print(f'Neg/query:   {NEGATIVES_PER_QUERY}')
print(f'Output:      {OUTPUT_DIR}')

In [ ]:
# ?? 3. Dataset + model ????????????????????????????????????????????????????????
import json, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer
from tqdm.notebook import tqdm

random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda'


class TripletDataset(Dataset):
    def __init__(self, paths, negatives_per_query=1):
        self.examples = []
        self.negatives_per_query = negatives_per_query
        for path in paths:
            if not Path(path).exists():
                print(f'WARNING: {path} nen?jden? ? preskakujem')
                continue
            with open(path, encoding='utf-8') as f:
                for line in f:
                    ex = json.loads(line)
                    self.examples.append((ex['anchor'], ex['positive'], ex['negatives']))
        random.shuffle(self.examples)
        print(f'Na??tan?ch {len(self.examples)} pr?kladov')

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        anchor, positive, negatives = self.examples[idx]
        k = min(self.negatives_per_query, len(negatives))
        sampled = random.sample(negatives, k=k)
        if k < self.negatives_per_query:
            sampled.extend(random.choices(negatives, k=self.negatives_per_query - k))
        return anchor, positive, sampled


def collate_triplets(batch):
    anchors, positives, negatives = zip(*batch)
    return list(anchors), list(positives), list(negatives)


data_paths = [DATA_PATH]
if USE_EXTRA and EXTRA_DATA.exists():
    data_paths.append(EXTRA_DATA)
    print(f'Kombinujem s: {EXTRA_DATA}')

dataset = TripletDataset(data_paths, negatives_per_query=NEGATIVES_PER_QUERY)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                     drop_last=True, num_workers=2, pin_memory=True,
                     collate_fn=collate_triplets)
print(f'{len(loader)} krokov/epochu')

print(f'\nNa??tavam {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

if N_GPUS >= 2:
    model = nn.DataParallel(model)
    print(f'DataParallel: {N_GPUS}x GPU ? batch {BATCH_SIZE} ? {BATCH_SIZE//N_GPUS}/GPU')

torch.backends.cudnn.benchmark = True
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'{n_params:.0f}M parametrov | VRAM cuda:0: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

In [ ]:
# ?? 4. Tr?ning ????????????????????????????????????????????????????????????????
def encode_batch(texts, max_length):
    enc = tokenizer(texts, padding=True, truncation=True,
                    max_length=max_length, return_tensors='pt').to(device)
    with torch.amp.autocast('cuda'):
        out = model(**enc)
    return F.normalize(out.last_hidden_state[:, 0], dim=-1)


def mnrl_loss(emb_a, emb_docs):
    # emb_docs: [(1 + k)B, dim] ? prv?ch B = positives, zvy?ok = sampled hard negatives
    # In-batch negatives: ka?d? anchor s?per? s B-1 cudz?ch positives + kB hard negs
    scores = emb_a @ emb_docs.T
    labels = torch.arange(scores.size(0), device=device)
    return F.cross_entropy(scores * 20.0, labels)


total_steps  = (len(loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = max(1, int(total_steps * 0.1))

try:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, fused=True)
    print('fused AdamW')
except Exception:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,
    lambda s: s / max(1, warmup_steps) if s < warmup_steps
              else max(0.0, 1.0 - (s - warmup_steps) / max(1, total_steps - warmup_steps)))
scaler = torch.amp.GradScaler('cuda')

print(f'Tr?ning: {EPOCHS} epochy ? {len(loader)} krokov = {total_steps} spolu')
print(f'Negatives/query: {NEGATIVES_PER_QUERY}')
print(f'Warmup: {warmup_steps} krokov')

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for step, (anchors, positives, negatives) in enumerate(pbar):
        B = len(anchors)
        flat_negatives = [neg for negs in negatives for neg in negs]
        emb_a  = encode_batch(anchors, ANCHOR_MAX_LEN)
        emb_pn = encode_batch(positives + flat_negatives, PASSAGE_MAX_LEN)
        # emb_pn: [(1 + k)B, dim] ? first B positives, zvy?ok sampled negatives
        loss = mnrl_loss(emb_a, emb_pn) / GRAD_ACCUM
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        epoch_loss += loss.item() * GRAD_ACCUM
        if (step + 1) % LOG_EVERY == 0:
            pbar.set_postfix(
                loss=f'{loss.item()*GRAD_ACCUM:.4f}',
                lr=f'{scheduler.get_last_lr()[0]:.2e}',
            )

    elapsed = (time.time() - t0) / 60
    print(f'Epoch {epoch}: loss={epoch_loss/len(loader):.4f} | {elapsed:.1f} min')

print(f'\nHotovo! {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── 5. Ulož model (sentence-transformers formát) ──────────────────────────────
from sentence_transformers import SentenceTransformer
from sentence_transformers import models as st_models

# Rozbaľ DataParallel wrapper ak bol použitý
base = model.module if isinstance(model, nn.DataParallel) else model
raw_model = getattr(base, '_orig_mod', base)

# HuggingFace checkpoint
hf_path = OUTPUT_DIR / 'hf_checkpoint'
raw_model.save_pretrained(str(hf_path))
tokenizer.save_pretrained(str(hf_path))
print(f'HF checkpoint → {hf_path}')

# Obal do sentence-transformers formátu (potrebné pre pipeline)
word_embedding = st_models.Transformer(str(hf_path))
pooling        = st_models.Pooling(word_embedding.get_embedding_dimension(), pooling_mode='cls')
st_model       = SentenceTransformer(modules=[word_embedding, pooling])
final_path     = OUTPUT_DIR / 'final'
st_model.save(str(final_path))
print(f'ST model → {final_path}')

In [ ]:
# ── 6. Výstup + publikuj ako dataset ─────────────────────────────────────────
from pathlib import Path

total_gb = sum(f.stat().st_size for f in OUTPUT_DIR.rglob('*') if f.is_file()) / 1e9
print(f'Model uložený: {OUTPUT_DIR}  ({total_gb:.2f} GB)')
print()
print('── Ako publikovať model ako Kaggle dataset ──────────────────────────────')
print('MOŽNOSŤ A — cez Output tab (bez sťahovania):')
print('  1. Notebook → Output tab (vpravo hore)')
print('  2. Klikni "New Dataset" vedľa e5-large-finetuned-v2/')
print('  3. Pomenuj: ct26-e5-finetuned-v2 → Create')
print()
print('MOŽNOSŤ B — spusti nasledujúcu bunku (Kaggle API automaticky)')

In [ ]:
# ── 7. (Voliteľné) Automaticky publikuj cez Kaggle API ───────────────────────
import json, os, subprocess
from pathlib import Path

# Kaggle notebooks majú KAGGLE_USERNAME a KAGGLE_KEY automaticky nastavené
username = os.environ.get('KAGGLE_USERNAME', '')
if not username:
    raise EnvironmentError('KAGGLE_USERNAME nie je nastavené — použi Možnosť A (Output tab)')

DATASET_NAME = 'ct26-e5-finetuned-v2'
dataset_id   = f'{username}/{DATASET_NAME}'

# Vytvor metadata súbor (potrebný pre kaggle API)
meta = {
    'title': DATASET_NAME,
    'id': dataset_id,
    'licenses': [{'name': 'CC0-1.0'}],
}
meta_path = OUTPUT_DIR / 'dataset-metadata.json'
meta_path.write_text(json.dumps(meta, indent=2))
print(f'Metadata: {meta_path}')

# Skús vytvoriť nový dataset; ak existuje, vytvor novú verziu
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(OUTPUT_DIR), '--dir-mode', 'zip'],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f'Dataset vytvorený: kaggle.com/{dataset_id}')
else:
    print('Dataset existuje — vytváram novú verziu...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(OUTPUT_DIR), '-m', 'v2 iterative hard negative mining'],
        capture_output=True, text=True,
    )
    print(result.stdout or result.stderr)

print(f'\nV pipeline notebooku (bunka 3) zmeň:')
print(f'  BASE = Path("/kaggle/input/{username}/{DATASET_NAME}")')